# Build 30-Paper Evaluation Sample

Create a reproducible 30-paper evaluation subset from `deepreview_test_2025.csv`.

**Sampling design (30 papers, seed = 42):**

| Stratum | Accept | Reject | What it captures |
|---|---|---|---|
| **Controversial** (`rating_std > 1.0`) | 5 | 5 | Reviewers disagreed on scores (final Accept/Reject still applies) |
| **Normal** (`rating_std ≤ 1.0`) | 10 | 10 | Reviewers largely agreed on scores |

All papers must have **≥ 4 human reviewers**. `rating_std` is the population standard deviation of human reviewer scores.

We sample each of the four buckets independently (same seed in each draw), then combine. Each paper is tagged with `stratum: "controversial" | "normal"` in the JSON output.

**Output:** `eval_sample_30.json`

## Setup

In [1]:
import csv
import json
import statistics
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

csv.field_size_limit(10_000_000)

DATA_DIR = Path(".")
SOURCE_CSV = DATA_DIR / "deepreview_test_2025.csv"
OUTPUT_JSON = DATA_DIR / "eval_sample_30.json"

SEED = 42
MIN_REVIEWERS = 4
CONTROVERSIAL_STD = 1.0  # rating_std above this → "controversial"

# Per-bucket sample sizes (must sum to 30)
N_CONTRO_ACCEPT = 5
N_CONTRO_REJECT = 5
N_NORMAL_ACCEPT = 10
N_NORMAL_REJECT = 10
N_SAMPLES = N_CONTRO_ACCEPT + N_CONTRO_REJECT + N_NORMAL_ACCEPT + N_NORMAL_REJECT

## Load source CSV

In [2]:
JSON_COLUMNS = ("inputs", "outputs", "rating", "reviewer_comments")


def load_deepreview_csv(path: Path) -> pd.DataFrame:
    rows = []
    with path.open(newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            for col in JSON_COLUMNS:
                row[col] = json.loads(row[col])
            rows.append(row)
    return pd.DataFrame(rows)


df = load_deepreview_csv(SOURCE_CSV)
print(f"Loaded {len(df):,} rows from {SOURCE_CSV.name}")
print(df[["id", "year", "mode", "decision"]].head())

Loaded 634 rows from deepreview_test_2025.csv
           id  year  mode decision
0  68J0pJFCi3  2025  best   Reject
1  JBgBrnhLLL  2025  best   Reject
2  OYT7yZfBFw  2025  best   Reject
3  breVfEOZLv  2025  best   Reject
4  960Ny6IjEr  2025  best   Reject


## Stratified sample

Split eligible papers into four buckets (stratum × decision), draw from each bucket with `random_state=SEED`, then combine.

In [3]:
def rating_std(ratings: list) -> float | None:
    if len(ratings) < 2:
        return None
    return statistics.pstdev(ratings)


def sample_bucket(pool: pd.DataFrame, n: int, bucket_name: str) -> pd.DataFrame:
    if len(pool) < n:
        raise ValueError(f"{bucket_name}: need {n} papers but only {len(pool)} available")
    stratum = bucket_name.split("/")[0]
    return pool.sample(n=n, random_state=SEED).assign(stratum=stratum)


df["n_reviewers"] = df["reviewer_comments"].apply(len)
df["rating_std"] = df["rating"].apply(rating_std)

eligible = df[(df["n_reviewers"] >= MIN_REVIEWERS) & df["rating_std"].notna()].copy()
controversial = eligible[eligible["rating_std"] > CONTROVERSIAL_STD]
normal = eligible[eligible["rating_std"] <= CONTROVERSIAL_STD]

buckets = {
    "controversial/Accept": (controversial[controversial["decision"] == "Accept"], N_CONTRO_ACCEPT),
    "controversial/Reject": (controversial[controversial["decision"] == "Reject"], N_CONTRO_REJECT),
    "normal/Accept": (normal[normal["decision"] == "Accept"], N_NORMAL_ACCEPT),
    "normal/Reject": (normal[normal["decision"] == "Reject"], N_NORMAL_REJECT),
}

print(f"Eligible papers (≥ {MIN_REVIEWERS} reviewers): {len(eligible):,} / {len(df):,}")
print("Pool sizes:")
for bucket_name, (pool, n) in buckets.items():
    print(f"  {bucket_name:22}: {len(pool):4} available → sample {n}")

sample_df = (
    pd.concat([sample_bucket(pool, n, name) for name, (pool, n) in buckets.items()])
    .sort_values("id")
    .reset_index(drop=True)
)

print(f"\nSampled {len(sample_df)} papers (seed={SEED})")
print("\nBy stratum × decision:")
print(sample_df.groupby(["stratum", "decision"]).size().to_string())
print(sample_df[["id", "stratum", "decision", "rating_std"]].head(10))

Eligible papers (≥ 4 reviewers): 544 / 634
Pool sizes:
  controversial/Accept  :   71 available → sample 5
  controversial/Reject  :  188 available → sample 5
  normal/Accept         :   97 available → sample 10
  normal/Reject         :  188 available → sample 10

Sampled 30 papers (seed=42)

By stratum × decision:
stratum        decision
controversial  Accept       5
               Reject       5
normal         Accept      10
               Reject      10
           id        stratum decision  rating_std
0  1Qpt43cqhg         normal   Accept    1.000000
1  6jxUsDAdAu         normal   Accept    1.000000
2  7EhS3YBxjY         normal   Accept    0.000000
3  85WHuB5CUK         normal   Reject    0.000000
4  8dzKkeWUUb  controversial   Accept    1.089725
5  FXJm5r17Q7  controversial   Reject    1.785357
6  GjM61KRiTG         normal   Accept    0.866025
7  Gx04TnVjee  controversial   Accept    1.299038
8  KgiMUvJcwm         normal   Reject    0.866025
9  KlN00vQEY2         normal   Accept 

## Export JSON

In [4]:
def extract_paper_text(inputs: list[dict]) -> str:
    for message in inputs:
        if message.get("role") == "user":
            return message["content"]
    raise ValueError("No user message found in inputs")


def row_to_record(row: pd.Series) -> dict:
    return {
        "id": row["id"],
        "year": row["year"],
        "mode": row["mode"],
        "decision": row["decision"],
        "stratum": row["stratum"],
        "ratings": row["rating"],
        "rating_std": row["rating_std"],
        "n_reviewers": int(row["n_reviewers"]),
        "paper_text": extract_paper_text(row["inputs"]),
        "human_reviews": row["reviewer_comments"],
    }


payload = {
    "metadata": {
        "created_at": datetime.now(timezone.utc).isoformat(),
        "source_csv": SOURCE_CSV.name,
        "seed": SEED,
        "n_samples": N_SAMPLES,
        "sampling_design": {
            "min_reviewers": MIN_REVIEWERS,
            "controversial_std_threshold": CONTROVERSIAL_STD,
            "buckets": {
                "controversial_accept": N_CONTRO_ACCEPT,
                "controversial_reject": N_CONTRO_REJECT,
                "normal_accept": N_NORMAL_ACCEPT,
                "normal_reject": N_NORMAL_REJECT,
            },
        },
        "eligible_count": len(eligible),
        "stratum_counts": {
            f"{stratum}_{decision.lower()}": int(count)
            for (stratum, decision), count in sample_df.groupby(["stratum", "decision"]).size().items()
        },
        "paper_ids": sample_df["id"].tolist(),
    },
    "papers": [row_to_record(row) for _, row in sample_df.iterrows()],
}

with OUTPUT_JSON.open("w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

size_mb = OUTPUT_JSON.stat().st_size / 1_000_000
print(f"Wrote {OUTPUT_JSON} ({size_mb:.2f} MB)")
print(f"Paper IDs: {payload['metadata']['paper_ids'][:5]} ...")

Wrote eval_sample_30.json (2.18 MB)
Paper IDs: ['1Qpt43cqhg', '6jxUsDAdAu', '7EhS3YBxjY', '85WHuB5CUK', '8dzKkeWUUb'] ...


## Quick validation

In [ ]:
with OUTPUT_JSON.open(encoding="utf-8") as f:
    loaded = json.load(f)

assert loaded["metadata"]["seed"] == SEED
assert len(loaded["papers"]) == N_SAMPLES
assert len({p["id"] for p in loaded["papers"]}) == N_SAMPLES

stratum_counts = Counter((p["stratum"], p["decision"]) for p in loaded["papers"])
assert stratum_counts[("controversial", "Accept")] == N_CONTRO_ACCEPT
assert stratum_counts[("controversial", "Reject")] == N_CONTRO_REJECT
assert stratum_counts[("normal", "Accept")] == N_NORMAL_ACCEPT
assert stratum_counts[("normal", "Reject")] == N_NORMAL_REJECT

controversial_stds = [p["rating_std"] for p in loaded["papers"] if p["stratum"] == "controversial"]
normal_stds = [p["rating_std"] for p in loaded["papers"] if p["stratum"] == "normal"]
assert all(std > CONTROVERSIAL_STD for std in controversial_stds)
assert all(std <= CONTROVERSIAL_STD for std in normal_stds)

print("Stratum × decision:", dict(stratum_counts))
first = loaded["papers"][0]
print(f"First paper: {first['id']} | {first['stratum']} | {first['decision']} | std={first['rating_std']:.2f}")
print(f"Paper text preview:\n{first['paper_text'][:400]}...")

Stratum × decision: {('normal', 'Accept'): 10, ('normal', 'Reject'): 10, ('controversial', 'Accept'): 5, ('controversial', 'Reject'): 5}
First paper: 1Qpt43cqhg | normal | Accept | std=1.00
Paper text preview:
\title{Fully-inductive Node Classification on Arbitrary Graphs}

\begin{abstract}
One fundamental challenge in graph machine learning is generalizing to new graphs. Many existing methods following the inductive setup can generalize to test graphs with new structures, but assuming the feature and label spaces remain the same as the training ones. 
This paper introduces the fully-inductive setup, wh...
